In [0]:
%sql
-- 1. Aseguramos que la tabla no tenga basura previa
-- DROP TABLE IF EXISTS workspace.products.silver_products;

In [0]:
%sql
DESCRIBE TABLE workspace.products.catalog;

In [0]:
%sql
DESCRIBE TABLE workspace.products.bronze_scraped_products_clean;

In [0]:
%sql
-- ALTER TABLE workspace.products.silver_products SET
-- TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported')

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS workspace.products.silver_products (
    product_id STRING,
    retailer STRING,
    brand STRING,
    name STRING,
    list_price DOUBLE,
    cash_price DOUBLE,
    scraped_at TIMESTAMP,
    scraped_date DATE,
    discount_pct DOUBLE,
    installments_json STRING,
    specification_json STRING,
    description STRING,
    rating STRING,
    stock STRING,
    is_available BOOLEAN,
    is_valid_name BOOLEAN,
    is_valid_price BOOLEAN,
    discount_applied_str STRING,
    discount_applied_int DOUBLE,
    has_installments BOOLEAN,
    has_description BOOLEAN
) 
USING DELTA 
PARTITIONED BY (scraped_date, retailer);

ALTER TABLE workspace.products.silver_products SET
TBLPROPERTIES('delta.feature.allowColumnDefaults' = 'supported');

In [0]:
%sql
DESCRIBE TABLE workspace.products.silver_products;

In [0]:
%sql
    
MERGE INTO workspace.products.silver_products AS target
USING (
  -- Aquí procesamos los datos de Bronze antes de insertarlos
  SELECT 
    b.product_id,
    b.retailer,
    c.brand, -- Usamos la marca del catálogo que es más confiable
    b.name,
    TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) AS list_price,
    TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) AS cash_price,
    b.scraped_at,
    CAST(b.scraped_at AS DATE) AS scraped_date,
    -- discount_pct calculado
    ROUND((1 - (TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2) AS discount_pct,
    b.installments AS installments_json,
    b.specifications AS specification_json,
    b.description,
    b.rating,
    b.is_in_stock AS stock,
    -- Columnas computadas (de celdas 8-14)
    CASE 
      WHEN TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
       AND TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
      THEN FALSE 
      ELSE TRUE 
    END AS is_available,
    CASE 
      WHEN b.name IS NULL OR b.name = '' THEN FALSE 
      ELSE TRUE 
    END AS is_valid_name,
    CASE 
      WHEN TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
       AND TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) IS NULL THEN FALSE
      WHEN TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) > 10000000 
       AND TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) > 10000000 THEN FALSE
      WHEN TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) < 0 
       AND TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) < 0 THEN FALSE
      ELSE TRUE 
    END AS is_valid_price,
    CASE 
      WHEN TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) = 0 THEN '0'
      ELSE CONCAT(ROUND((1 - (TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2), '%')
    END AS discount_applied_str,
    CASE 
      WHEN TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) IS NULL 
        OR TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE) = 0 THEN 0.0
      ELSE ROUND((1 - (TRY_CAST(REGEXP_REPLACE(b.cash_price, '[\\$.]', '') AS DOUBLE) / TRY_CAST(REGEXP_REPLACE(b.list_price, '[\\$.]', '') AS DOUBLE))) * 100, 2)
    END AS discount_applied_int,
    CASE
      WHEN (LOWER(b.installments) LIKE '%sin interés%' OR LOWER(b.installments) LIKE '%cuotas fijas%') THEN TRUE
      WHEN ARRAY_MAX(
             TRANSFORM(
               REGEXP_EXTRACT_ALL(b.installments, ':\\s*"*(\\d+)', 1), 
               x -> CAST(x AS INT)
             )
           ) > 1 THEN TRUE
      ELSE FALSE
    END AS has_installments,
    CASE 
      WHEN b.description IS NULL OR b.description = '' THEN FALSE 
      ELSE TRUE 
    END AS has_description
  FROM workspace.products.bronze_scraped_products_clean b
  INNER JOIN workspace.products.catalog c 
    ON b.product_id = c.product_id AND b.retailer = c.retailer
) AS source
ON target.product_id = source.product_id 
   AND target.retailer = source.retailer 
   AND target.scraped_at = source.scraped_at
-- Si ya existe esa combinación exacta, podrías actualizar (opcional)
WHEN MATCHED THEN
  UPDATE SET 
    target.list_price = source.list_price,
    target.cash_price = source.cash_price,
    target.discount_pct = source.discount_pct,
    target.installments_json = source.installments_json,
    target.specification_json = source.specification_json,
    target.description = source.description,
    target.rating = source.rating,
    target.stock = source.stock,
    target.is_available = source.is_available,
    target.is_valid_name = source.is_valid_name,
    target.is_valid_price = source.is_valid_price,
    target.discount_applied_str = source.discount_applied_str,
    target.discount_applied_int = source.discount_applied_int,
    target.has_installments = source.has_installments,
    target.has_description = source.has_description
-- Si no existe, es un dato nuevo del scraping de hoy y se inserta
WHEN NOT MATCHED THEN
  INSERT (
    product_id, retailer, brand, name, list_price, cash_price, scraped_at, scraped_date, discount_pct, 
    installments_json, specification_json, description, rating, stock,
    is_available, is_valid_name, is_valid_price, discount_applied_str, discount_applied_int, 
    has_installments, has_description
  )
  VALUES (
    source.product_id, 
    source.retailer, 
    source.brand, 
    source.name, 
    source.list_price, 
    source.cash_price, 
    source.scraped_at, 
    source.scraped_date,
    source.discount_pct,
    source.installments_json,
    source.specification_json,
    source.description,
    source.rating,
    source.stock,
    source.is_available,
    source.is_valid_name,
    source.is_valid_price,
    source.discount_applied_str,
    source.discount_applied_int,
    source.has_installments,
    source.has_description
  )